In [ ]:
#Week 6, Day 6 — June 27, 2026
#Build Spatial Risk Map Component in Dashboard
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd, numpy as np, matplotlib.pyplot as plt
import seaborn as sns, os, json, warnings, datetime
warnings.filterwarnings('ignore')
BASE_DIR = "/content/drive/MyDrive/HCLc_Project"
DIRS = {"raw":BASE_DIR+"/data/raw","processed":BASE_DIR+"/data/processed",
        "metadata":BASE_DIR+"/data/metadata","visualizations":BASE_DIR+"/visualizations"}
print("Environment ready ")

import folium
from folium.plugins import HeatMap
# Load all required datasets
master     = pd.read_csv(DIRS['processed']+'/master_table_v2.csv')
cluster_df = pd.read_csv(DIRS['processed']+'/cluster_results.csv')
hotspots   = pd.read_csv(DIRS['processed']+'/hotspots.csv')
df_c       = pd.read_csv(DIRS['processed']+'/climate_clean_v1.csv', parse_dates=['Date'])
df_c['year']  = df_c['Date'].dt.year
df_c['month'] = df_c['Date'].dt.month

# Attach risk labels and silhouette scores from K-Means (Week 5)
df_c_clean = df_c.dropna(subset=['SST (°C)','pH Level','Species Observed']).reset_index(drop=True)
df_c_clean['Risk_Cluster']     = cluster_df['Risk_Cluster'].values
df_c_clean['Silhouette_Score'] = cluster_df['Silhouette_Score'].values

print(f'Climate records (for map markers) : {len(df_c_clean)}')
print(f'Plastic hotspot cells             : {len(hotspots)}')
print(f'Master table rows                 : {len(master)}')
print()
print('Climate dataset locations:')
print(df_c_clean.groupby('Location')[['Latitude','Longitude','Risk_Cluster']].agg(
    {'Latitude':'mean','Longitude':'mean','Risk_Cluster':'first'}).round(2).to_string())
#Step 1: Initialize Folium Map
# Centre map on North Indian Ocean / global view showing all climate locations
m = folium.Map(
    location=[10, 60],     # centred between Indian Ocean and climate dataset locations
    zoom_start=3,
    tiles='CartoDB positron',   # clean light-grey base map
    prefer_canvas=True,
)

print('Folium map initialized ')
print(f'  Centre    : [10, 60]')
print(f'  Zoom      : 3  (shows Indian Ocean + all climate monitoring stations)')
print(f'  Base tiles: CartoDB positron (clean, mentor-friendly)')
#Step 2: Layer 1 — Risk Zone Circle Markers (500 markers)
risk_colors_f = {'Stable':'green', 'At_Risk':'orange', 'Critical':'red'}

# Create a named FeatureGroup so the layer can be toggled on/off
risk_group = folium.FeatureGroup(name=' Risk Zones (SST/pH/Species)', show=True)

for _, row in df_c_clean.iterrows():
    risk  = row['Risk_Cluster']
    color = risk_colors_f.get(risk, 'blue')

    # Build clickable popup with full data
    popup_html = (
        f"<b>Location:</b> {row.get('Location','—')}<br>"
        f"<b> Date:</b> {str(row['Date'])[:10]}<br>"
        f"<hr style='margin:4px'>"
        f"<b> SST:</b> {row['SST (°C)']:.2f}°C<br>"
        f"<b> pH:</b> {row['pH Level']:.4f}<br>"
        f"<b> Species Observed:</b> {int(row['Species Observed'])}<br>"
        f"<b> Bleaching:</b> {row.get('Bleaching Severity','—')}<br>"
        f"<hr style='margin:4px'>"
        f"<b style='color:{color}'> Risk Zone: {risk}</b><br>"
        f"<b> Silhouette Score:</b> {row['Silhouette_Score']:.4f}"
    )

    folium.CircleMarker(
        location  = [row['Latitude'], row['Longitude']],
        radius    = 6,
        color     = color,
        fill      = True,
        fill_color= color,
        fill_opacity = 0.70,
        popup  = folium.Popup(popup_html, max_width=270),
        tooltip= f"{risk} — SST {row['SST (°C)']:.1f}°C | Species {int(row['Species Observed'])}",
    ).add_to(risk_group)

risk_group.add_to(m)

# Layer breakdown
risk_counts = df_c_clean['Risk_Cluster'].value_counts()
print(f'Layer 1 added: {len(df_c_clean)} risk zone markers ')
for risk, color in risk_colors_f.items():
    n = risk_counts.get(risk, 0)
    print(f'  {color:<8} {risk:<12}: {n} markers')
#Step 3: Layer 2 — Plastic Hotspot Fire Icons (31 markers)
hotspot_group = folium.FeatureGroup(name='Plastic Hotspots (Top 20% Density)', show=True)

for _, h in hotspots.iterrows():
    popup_html = (
        f"<b>Plastic Hotspot</b><br>"
        f"<hr style='margin:4px'>"
        f"<b>Grid Cell:</b> ({h['grid_lat']}°N, {h['grid_lon']}°E)<br>"
        f"<b>Ocean Zone:</b> {h.get('Ocean_Zone','—')}<br>"
        f"<b>Plastic Density:</b> {h['Plastic_Density_kg_km2']:.4f} kg/km²<br>"
        f"<b>Record Count:</b> {h.get('Plastic_Record_Count', '—')}<br>"
        f"<b>Threshold:</b> Top 20% (≥448.73 kg/cell)"
    )

    folium.Marker(
        location = [h['grid_lat'] + 0.5, h['grid_lon'] + 0.5],
        popup    = folium.Popup(popup_html, max_width=240),
        icon     = folium.Icon(color='red', icon='fire', prefix='fa'),
        tooltip  = f"Hotspot | Zone: {h.get('Ocean_Zone','—')} | {h['Plastic_Density_kg_km2']:.2f} kg/km²",
    ).add_to(hotspot_group)

hotspot_group.add_to(m)
print(f'Layer 2 added: {len(hotspots)} plastic hotspot markers ')
print(f'  Hotspot threshold : ≥448.73 kg per grid cell (80th percentile)')
print(f'  Icon              : Red fire icon (Font Awesome)')
print(f'  Popup             : Grid coordinates, zone, density, threshold reference')
#Step 4: Layer 3 — Plastic Density HeatMap
# Build heat data: [lat, lon, weight] for each plastic-bearing grid cell
plastic_rows = master.dropna(subset=['Plastic_Density_kg_km2'])
heat_data = [
    [row['grid_lat'] + 0.5, row['grid_lon'] + 0.5, row['Plastic_Density_kg_km2']]
    for _, row in plastic_rows.iterrows()
]

heatmap_group = folium.FeatureGroup(name=' Plastic Density Heatmap', show=True)

HeatMap(
    heat_data,
    radius    = 22,
    blur      = 16,
    min_opacity = 0.35,
    gradient  = {
        '0.2': 'blue',
        '0.4': 'cyan',
        '0.6': 'yellow',
        '0.8': 'orange',
        '1.0': 'red',
    },
).add_to(heatmap_group)

heatmap_group.add_to(m)
print(f'Layer 3 added: Plastic density HeatMap ')
print(f'  Grid cells in heatmap : {len(heat_data)}')
print(f'  Colour gradient       : blue (low) → cyan → yellow → orange → red (high)')
print(f'  Radius / blur         : 22 / 16  (visible at zoom 3–6)')

#Step 5: Layer Toggle Control + Legend
# Layer control — mentors can toggle any layer on/off
folium.LayerControl(
    position   = 'topright',
    collapsed  = False,    # always show layer panel expanded
).add_to(m)

# Floating legend box
legend_html = (
    "<div style='"
    "position:fixed;bottom:30px;left:30px;z-index:1000;"
    "background:white;padding:16px;"
    "border:2px solid #555;border-radius:10px;"
    "font-size:12px;line-height:2.0;"
    "box-shadow:3px 3px 8px rgba(0,0,0,0.3)'>"
    "<b> Risk Map Legend</b><br>"
    "<span style='color:green'>●</span> <b>Stable</b> — SST &lt;28°C, Species &gt;135<br>"
    "<span style='color:orange'>●</span> <b>At-Risk</b> — SST 28–30°C, Species 100–135<br>"
    "<span style='color:red'>●</span> <b>Critical</b> — SST &gt;30°C, Species &lt;100<br>"
    "<b> Hotspot</b> — Top 20% plastic density (≥448 kg)<br>"
    "<b> Heatmap</b> — Plastic density gradient overlay<br>"
    "<hr style='margin:4px'>"
    "<small>Click any marker for full data popup.<br>"
    "Use layer control (top-right) to toggle layers.</small>"
    "</div>"
)
m.get_root().html.add_child(folium.Element(legend_html))

# Save map
map_path = DIRS['visualizations'] + '/Week6_Dashboard_Spatial_Map.html'
m.save(map_path)
size_kb = os.path.getsize(map_path) / 1024

print(f'3-layer interactive Folium map saved ')
print(f'  File : Week6_Dashboard_Spatial_Map.html')
print(f'  Size : {size_kb:.1f} KB')
print()
print('MAP SUMMARY:')
print(f'  Layer 1 : {len(df_c_clean)} risk zone circle markers (green/orange/red)')
print(f'  Layer 2 : {len(hotspots)} plastic hotspot fire icons')
print(f'  Layer 3 : {len(heat_data)}-cell plastic density HeatMap')
print(f'  Control : Layer toggle panel (top-right, always expanded)')
print(f'  Legend  : Floating box (bottom-left)')
print()
print('INTERACTIVITY:')
print('   Click any circle marker → full popup (SST, pH, Species, Risk, Silhouette)')
print('   Click any fire icon     → popup (grid cell, zone, density, threshold)')
print('   Toggle layer panel      → show/hide each layer independently')
print('   Scroll / pinch to zoom  → zoom in to Arabian Sea, Bay of Bengal, etc.')

#Step 6: Static Spatial Map for Dashboard PNG Export
# Also save a static Matplotlib version for PDF reports and dashboard screenshots
fig, ax = plt.subplots(figsize=(14, 9))
ax.set_facecolor('#cce5ff')

# Background ocean colour blocks
import matplotlib.patches as mpatches
ocean_zones = [
    ('Arabian Sea',         (65, 5,  75-65, 20),  '#aad4f5'),
    ('Bay of Bengal',       (75, 5,  90-75, 22),  '#7fc4f0'),
    ('Andaman Sea',         (90, 10, 95-90, 20),  '#5ab5e8'),
    ('Gulf of Kutch',       (65, 22, 75-65, 8),   '#93c9e0'),
    ('Indian Ocean (gen)',  (75, 22, 20,    8),   '#a8d5e2'),
]
for name, (x,y,w,h), color in ocean_zones:
    ax.add_patch(mpatches.FancyBboxPatch((x,y), w, h,
        boxstyle='round,pad=0', facecolor=color, alpha=0.5, edgecolor='none'))

# Hotspot cells
for _, h in hotspots.iterrows():
    ax.scatter(h['grid_lon']+0.5, h['grid_lat']+0.5,
               c='#e74c3c', s=120, marker='*', zorder=6,
               edgecolors='darkred', linewidth=0.5)

# All plastic grid cells (non-hotspot)
plastic_rows = master.dropna(subset=['Plastic_Density_kg_km2'])
threshold_val = plastic_rows['Plastic_Density_kg_km2'].quantile(0.80)
non_hot = plastic_rows[plastic_rows['Plastic_Density_kg_km2'] < threshold_val]
sc = ax.scatter(non_hot['grid_lon']+0.5, non_hot['grid_lat']+0.5,
                c=non_hot['Plastic_Density_kg_km2'],
                cmap='YlOrRd', s=40, alpha=0.7, zorder=5,
                vmin=0, vmax=plastic_rows['Plastic_Density_kg_km2'].max())
plt.colorbar(sc, ax=ax, label='Plastic Density (kg/km²)', shrink=0.8)

# Species observation density (heat contour proxy)
bb_s_sample = master.dropna(subset=['Species_Count'])
ax.scatter(bb_s_sample['grid_lon']+0.5, bb_s_sample['grid_lat']+0.5,
           c='#27ae60', s=6, alpha=0.2, zorder=3, label='Species observations')

# Risk zone centroids
centroid_data = [
    ('Stable',   '#27ae60', 72.0, 10.0, 'SST 27.1°C\n141 sp.'),
    ('At-Risk',  '#f39c12', 80.0, 18.0, 'SST 28.6°C\n119 sp.'),
    ('Critical', '#e74c3c', 88.0, 25.0, 'SST 30.4°C\n96 sp.'),
]
for label, color, lon, lat, ann in centroid_data:
    ax.scatter(lon, lat, c=color, s=300, marker='D',
               edgecolors='black', linewidth=1.5, zorder=8, label=f'{label} zone')
    ax.annotate(ann, (lon, lat), xytext=(5,5), textcoords='offset points',
                fontsize=7, color=color, fontweight='bold')

# Legend items
hotspot_patch = mpatches.Patch(color='#e74c3c', label='🔥 Plastic Hotspot (top 20%)')
ax.legend(handles=[hotspot_patch]+ax.get_legend_handles_labels()[0],
          loc='lower right', fontsize=8, framealpha=0.9)

# Labels
ax.set_xlim(63, 97); ax.set_ylim(3, 31)
ax.set_xlabel('Longitude (°E)', fontsize=11)
ax.set_ylabel('Latitude (°N)',  fontsize=11)
ax.set_title('North Indian Ocean — Spatial Risk Map\n'
             ' Plastic Hotspots |  Density Gradient |  K-Means Risk Zone Centroids',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.3, linestyle='--')

# Ocean zone text labels
for name,(x,y,w,h),_ in ocean_zones:
    ax.text(x+w/2, y+h/2, name.replace(' ','\n'),
            ha='center', va='center', fontsize=7,
            color='#1a3a5c', alpha=0.7, style='italic')

plt.tight_layout()
static_path = DIRS['visualizations']+'/Week6_Day6_spatial_risk_map_static.png'
plt.savefig(static_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Static spatial risk map saved   ({os.path.getsize(static_path)/1024:.1f} KB)')